# File Operations and Processing

This notebook demonstrates file upload, processing, and retrieval in the ADEPT ecosystem.

## Security Architecture

All file operations route through `nginx_proxy` (TLS + JWT authentication).
JupyterLab is isolated on `frontend_net` — it cannot reach internal services directly.

## Features Covered
- Upload files via nginx_proxy API
- Process files with AI agents
- File registration and metadata
- Download processed results
- Working with CSV, JSON, and text files

## Docker Configuration
- JupyterLab service: `deployment/local/docker-compose/docker-compose.core.yaml`
- Rebuild after config changes: `make rebuild-jupyterlab`

## 1. Setup and Authentication

In [ ]:
import os
import json
import httpx
import asyncio
import pandas as pd
from pathlib import Path
import openai

# Load authentication
def load_auth():
    with open("/tmp/adept_auth.json", "r") as f:
        return json.load(f)

def load_config():
    with open("/tmp/adept_config.json", "r") as f:
        return json.load(f)

auth = load_auth()
config = load_config()

API_BASE_URL = config.get('api_base_url', os.getenv('ADEPT_API_BASE_URL', 'https://nginx_proxy/v1'))
NGINX_URL = config.get('nginx_url', 'https://nginx_proxy')
ACCESS_TOKEN = auth['access_token']

# Initialize OpenAI client — routes through nginx_proxy
client = openai.OpenAI(
    base_url=API_BASE_URL,
    api_key=ACCESS_TOKEN,
    http_client=httpx.Client(verify=False)
)

print(f"Connected to: {API_BASE_URL}")
print(f"User: {auth['username']}")
print(f"Routing: nginx_proxy (TLS + JWT)")

## 2. Create Sample Data Files

In [ ]:
# Create sample CSV data
import tempfile

# Sample CSV - experimental data
csv_data = pd.DataFrame({
    'sample_id': ['S001', 'S002', 'S003', 'S004', 'S005'],
    'temperature': [37.2, 38.1, 36.9, 37.5, 38.3],
    'concentration': [2.5, 3.1, 2.8, 2.9, 3.3],
    'ph_level': [7.2, 7.4, 7.1, 7.3, 7.5],
    'reaction_time': [120, 135, 125, 130, 140]
})

csv_file = "/tmp/experiment_data.csv"
csv_data.to_csv(csv_file, index=False)
print(f"✅ Created: {csv_file}")
print(csv_data.head())

# Sample JSON - configuration data
json_data = {
    "experiment": {
        "name": "Protein Binding Analysis",
        "date": "2024-01-15",
        "parameters": {
            "temperature": 37,
            "duration": 120,
            "buffer": "PBS"
        },
        "samples": 5
    }
}

json_file = "/tmp/experiment_config.json"
with open(json_file, "w") as f:
    json.dump(json_data, f, indent=2)
print(f"\n✅ Created: {json_file}")
print(json.dumps(json_data, indent=2))

## 3. Upload Files

In [ ]:
async def upload_file(file_path: str) -> dict:
    """Upload a file via nginx_proxy."""
    async with httpx.AsyncClient(verify=False, timeout=30.0) as http_client:
        with open(file_path, "rb") as f:
            files = {"file": (Path(file_path).name, f, "application/octet-stream")}
            data = {"purpose": "assistants"}
            headers = {"Authorization": f"Bearer {ACCESS_TOKEN}"}
            
            response = await http_client.post(
                f"{API_BASE_URL}/files",
                files=files,
                data=data,
                headers=headers
            )
            
            if response.status_code in [200, 201]:
                return response.json()
            else:
                return {"error": f"Upload failed: {response.text}"}

# Upload both files
print("Uploading files...\n")

csv_upload = await upload_file(csv_file)
if "error" not in csv_upload:
    print(f"CSV uploaded: {csv_upload['id']}")
    print(f"   Filename: {csv_upload['filename']}")
    print(f"   Size: {csv_upload['bytes']} bytes")
else:
    print(f"CSV upload failed: {csv_upload['error']}")

json_upload = await upload_file(json_file)
if "error" not in json_upload:
    print(f"\nJSON uploaded: {json_upload['id']}")
    print(f"   Filename: {json_upload['filename']}")
    print(f"   Size: {json_upload['bytes']} bytes")
else:
    print(f"JSON upload failed: {json_upload['error']}")

## 4. Process Files with Agent

In [ ]:
# Ask agent to analyze the uploaded files
if "error" not in csv_upload:
    prompt = f"""I've uploaded two files:
    1. {csv_upload['filename']} (ID: {csv_upload['id']}) - CSV with experimental data
    2. {json_upload['filename']} (ID: {json_upload['id']}) - JSON with experiment configuration
    
    Please analyze the CSV data and provide:
    1. Statistical summary of the measurements
    2. Any correlations between variables
    3. Recommendations based on the data
    """
    
    print("Sending to agent for analysis...\n")
    
    response = client.chat.completions.create(
        model="agentic-framework:scientific-agent-v1",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7,
        max_tokens=1500
    )
    
    print("Agent Analysis:")
    print("=" * 60)
    print(response.choices[0].message.content)

## 5. List Uploaded Files

In [ ]:
async def list_files():
    """List all uploaded files via nginx_proxy."""
    async with httpx.AsyncClient(verify=False, timeout=30.0) as http_client:
        headers = {"Authorization": f"Bearer {ACCESS_TOKEN}"}
        response = await http_client.get(
            f"{API_BASE_URL}/files",
            headers=headers
        )
        
        if response.status_code == 200:
            return response.json()
        else:
            return {"error": f"Failed to list files: {response.text}"}

# List files
files_list = await list_files()

if "error" not in files_list:
    print(f"\nFiles in your account: {len(files_list.get('data', []))}\n")
    
    for file in files_list.get('data', [])[:5]:
        print(f"File: {file['filename']}")
        print(f"  ID: {file['id']}")
        print(f"  Size: {file['bytes']} bytes")
        print(f"  Created: {file['created_at']}")
        print()
else:
    print(f"Error: {files_list['error']}")

## 6. Download File Content

In [ ]:
async def download_file(file_id: str, output_path: str = None):
    """Download file content via nginx_proxy."""
    async with httpx.AsyncClient(verify=False, timeout=30.0) as http_client:
        headers = {"Authorization": f"Bearer {ACCESS_TOKEN}"}
        response = await http_client.get(
            f"{API_BASE_URL}/files/{file_id}/content",
            headers=headers
        )
        
        if response.status_code == 200:
            if output_path:
                with open(output_path, "wb") as f:
                    f.write(response.content)
                return {"success": True, "path": output_path}
            else:
                return {"success": True, "content": response.content}
        else:
            return {"error": f"Download failed: {response.text}"}

# Download the CSV file we uploaded
if "error" not in csv_upload:
    download_path = "/tmp/downloaded_data.csv"
    result = await download_file(csv_upload['id'], download_path)
    
    if result.get('success'):
        print(f"Downloaded to: {result['path']}")
        df = pd.read_csv(download_path)
        print("\nDownloaded data:")
        print(df.head())
    else:
        print(f"Download failed: {result.get('error')}")

## 7. Generate and Download Results

In [ ]:
# Ask agent to generate a plot from the data
plot_prompt = f"""Using the CSV file {csv_upload['filename']} (ID: {csv_upload['id']}),
please create a correlation matrix heatmap showing relationships between all numerical variables.
Save the plot and provide the file ID so I can download it."""

print("Requesting plot generation...\n")

plot_response = client.chat.completions.create(
    model="agentic-framework:scientific-agent-v1",
    messages=[{"role": "user", "content": plot_prompt}],
    temperature=0.7,
    max_tokens=1000
)

print("Agent Response:")
print("=" * 60)
print(plot_response.choices[0].message.content)

## 8. Clean Up

In [ ]:
async def delete_file(file_id: str):
    """Delete a file via nginx_proxy."""
    async with httpx.AsyncClient(verify=False, timeout=30.0) as http_client:
        headers = {"Authorization": f"Bearer {ACCESS_TOKEN}"}
        response = await http_client.delete(
            f"{API_BASE_URL}/files/{file_id}",
            headers=headers
        )
        
        if response.status_code in [200, 204]:
            return {"success": True}
        else:
            return {"error": f"Delete failed: {response.text}"}

# Optional: Delete uploaded files (uncomment to clean up)
# if "error" not in csv_upload:
#     result = await delete_file(csv_upload['id'])
#     if result.get('success'):
#         print(f"Deleted CSV file: {csv_upload['id']}")

print("Files preserved for future use.")
print("Uncomment deletion code to clean up.")

## Summary

You've learned how to:
- ✅ Upload files to the Agent Gateway
- ✅ Reference files in agent conversations
- ✅ List and manage uploaded files
- ✅ Download file content
- ✅ Process data with AI agents
- ✅ Generate visualizations from data

### Next Steps
- Try uploading different file types (PDF, images)
- Use the CSV RAG tool for data analysis
- Combine multiple files in analysis
- Explore the `03_scientific_workflows.ipynb` notebook